In [ ]:
from pathlib import Path
import sys
import numpy as np

BASE_DIR = Path().cwd().parent
PICTURE_DATA_DIR = BASE_DIR / "picture_data"
LEUKEMIA_CANCER_DATA_DIR = PICTURE_DATA_DIR / "cancer_images"
DATASET_DIR = LEUKEMIA_CANCER_DATA_DIR / "cancer_analysis"
TRAIN_DIR = DATASET_DIR / "train"
VALIDATION_DIR = DATASET_DIR / "validation"
TEST_DIR = DATASET_DIR / "test"

AUGMENTED_DIR = LEUKEMIA_CANCER_DATA_DIR / "augmented_cancer"

FINE_TUNING_DIR = BASE_DIR / "leukemia_cancer"
FEATURE_DIR = FINE_TUNING_DIR / "features"

CLASS_NAMES = [
    "Benign",
    "Early",
    "Pre",
    "Pro"
]

VERSION = 3

MODEL_TYPE = "densenet121"

# Version 1: Resnet50 ohne Augmentation
# Version 2: Resnet50 mit Augmentation
# Version 3: Densenet121 mit Augmentation

sys.path.append(str(BASE_DIR))

from data_handling.manage_data import create_dataset, extract_features
from model.build_model import split_base_model

In [2]:
train_ds = create_dataset(TRAIN_DIR, CLASS_NAMES, batch_size=32, model_type=MODEL_TYPE)
validation_ds = create_dataset(VALIDATION_DIR, CLASS_NAMES, batch_size=16, model_type=MODEL_TYPE)
test_ds = create_dataset(TEST_DIR, CLASS_NAMES, batch_size=16, model_type=MODEL_TYPE)

train_ds_augmented = create_dataset(AUGMENTED_DIR, CLASS_NAMES, batch_size=32, model_type=MODEL_TYPE)

train_ds = train_ds.concatenate(train_ds_augmented)

Found 1952 files belonging to 4 classes.
Found 814 files belonging to 4 classes.
Found 490 files belonging to 4 classes.
Found 5856 files belonging to 4 classes.


In [3]:
feature_extractor, _ = split_base_model(model_type=MODEL_TYPE)

In [4]:
X_train, y_train = extract_features(feature_extractor, train_ds)
X_validation, y_validation = extract_features(feature_extractor, validation_ds)
X_test, y_test = extract_features(feature_extractor, test_ds)

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    FEATURE_DIR / f"features_{VERSION}.npz",
    X_train=X_train,
    y_train=y_train,
    X_validation=X_validation,
    y_validation=y_validation,
    X_test=X_test,
    y_test=y_test
)